In [63]:
from IPython.core.display import HTML
from IPython.display import display
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [64]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [65]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [66]:
from sklearn.model_selection import train_test_split

X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

## Data Preprocessing

In [67]:
import string
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

print(string.punctuation)
print(stopwords.words("english")[100:110])
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [68]:
import re

pattern = re.compile("[%s]" % re.escape(string.punctuation))


def clean_html(html):
    cleaned = re.sub(r"(?is)<(script|style).*?>.*?(</\\1>)", " ", html)
    cleaned = re.sub(r"(?s)<!--(.*?)-->[\n]?", " ", cleaned)
    cleaned = re.sub(r"(?s)<.*?>", " ", cleaned)
    return cleaned.strip()

data["no_html"] = data["text"].apply(clean_html)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [69]:
def clean_text(text):
    text = pattern.sub(u"", text)

    # Remove all special characters (keep only letters, numbers, spaces)
    text = re.sub(r'\W', ' ', text)
    
    # Remove numbers
    text = re.sub(r'\d+', ' ', text)
    
    # Remove single characters from the start (e.g., "a word" → "word")
    text = re.sub(r'^[a-zA-Z]\s+', '', text)
    
    # Remove all single characters
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    
    # Substitute multiple spaces with single space
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove prefixed 'b' (e.g., b'hello' from byte string literals)
    text = re.sub(r"\bb'", '', text)
    
    # Convert to lowercase
    text = text.lower()
    
    return text

data["email_cleaned"] = data["no_html"].apply(clean_text)
data.head()

,text,label,no_html,email_cleaned
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",dear sir strictly private business proposal am...
1,Will do.,0,Will do.,will do
2,Nora--Cheryl has emailed dozens of memos about...,0,Nora--Cheryl has emailed dozens of memos about...,noracheryl has emailed dozens of memos about h...
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,Dear Sir=2FMadam=2C I know that this proposal ...,dear sir fmadam i know that this proposal migh...
4,fyi,0,fyi,fyi


## Now let's work on removing stopwords
Remove the stopwords.

In [70]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

data["email_cleaned_no_stopwords"] = data["email_cleaned"].apply(remove_stopwords)
data.head()

,text,label,no_html,email_cleaned,email_cleaned_no_stopwords
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",dear sir strictly private business proposal am...,dear sir strictly private business proposal mi...
1,Will do.,0,Will do.,will do,
2,Nora--Cheryl has emailed dozens of memos about...,0,Nora--Cheryl has emailed dozens of memos about...,noracheryl has emailed dozens of memos about h...,noracheryl emailed dozens memos haiti weekend ...
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,Dear Sir=2FMadam=2C I know that this proposal ...,dear sir fmadam i know that this proposal migh...,dear sir fmadam know proposal might surprise e...
4,fyi,0,fyi,fyi,fyi


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [71]:
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import wordnet
import nltk
#nltk.download('averaged_perceptron_tagger')

wordnet_lemma = WordNetLemmatizer()  


def lemmatize(text):
    words = text.split()
    words = [wordnet_lemma.lemmatize(word) for word in words]
    return " ".join(words)

data["lemmas"] = data["email_cleaned_no_stopwords"].apply(lemmatize)
data.tail()

,text,label,no_html,email_cleaned,email_cleaned_no_stopwords,lemmas
995,So what's the latest? It sounds contradictory ...,0,So what's the latest? It sounds contradictory ...,so whats the latest it sounds contradictory an...,whats latest sounds contradictory af decide sh...,whats latest sound contradictory af decide sha...
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",transfer of million pounds to youraccountmy na...,transfer million pounds youraccountmy name mre...,transfer million pound youraccountmy name mrej...
997,Barb I will call to explain. Are you back in t...,0,Barb I will call to explain. Are you back in t...,barb will call to explain are you back in the ...,barb call explain back country h,barb call explain back country h
998,Yang on travelNot free tonite.May work tomorrow,0,Yang on travelNot free tonite.May work tomorrow,yang on travelnot free tonitemay work tomorrow,yang travelnot free tonitemay work tomorrow,yang travelnot free tonitemay work tomorrow
999,sbwhoeopSunday February 21 2010 7:42 PMHShaunH...,0,sbwhoeopSunday February 21 2010 7:42 PMHShaunH...,sbwhoeopsunday february pmhshaunh just talked ...,sbwhoeopsunday february pmhshaunh talked shaun...,sbwhoeopsunday february pmhshaunh talked shaun...


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [72]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

ham_data = data[data["label"] == 1]
spam_data = data[data["label"] == 0]

X = vectorizer.fit_transform(data["lemmas"])

X_ham = vectorizer.transform(ham_data)
X_spam = vectorizer.transform(spam_data)

vocab = vectorizer.get_feature_names_out()

In [73]:
ham_counts = X_ham.sum(axis=0).A1
spam_counts = X_spam.sum(axis=0).A1

In [74]:
import numpy as np

top10_ham_idx = np.argsort(ham_counts)[::-1][:10]
top10_spam_idx = np.argsort(spam_counts)[::-1][:10]

print(f"Top 10 ham idx: {[vocab[id] for id in top10_ham_idx]}")
print(f"Top 10 spam idx: {[vocab[id] for id in top10_spam_idx]}")

Top 10 ham idx: ['text', 'â½ã', 'ghtvx', 'ghcv', 'ghdie', 'ghg', 'ghhfpwmtyvdt', 'ghhzbbc', 'ghkkxo', 'ghnhvxqma']
Top 10 spam idx: ['text', 'â½ã', 'ghtvx', 'ghcv', 'ghdie', 'ghg', 'ghhfpwmtyvdt', 'ghhzbbc', 'ghkkxo', 'ghnhvxqma']


In [75]:
data["preprocessed_text"] = data["lemmas"]
data.head()

,text,label,no_html,email_cleaned,email_cleaned_no_stopwords,lemmas,preprocessed_text
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",dear sir strictly private business proposal am...,dear sir strictly private business proposal mi...,dear sir strictly private business proposal mi...,dear sir strictly private business proposal mi...
1,Will do.,0,Will do.,will do,,,
2,Nora--Cheryl has emailed dozens of memos about...,0,Nora--Cheryl has emailed dozens of memos about...,noracheryl has emailed dozens of memos about h...,noracheryl emailed dozens memos haiti weekend ...,noracheryl emailed dozen memo haiti weekend pl...,noracheryl emailed dozen memo haiti weekend pl...
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,Dear Sir=2FMadam=2C I know that this proposal ...,dear sir fmadam i know that this proposal migh...,dear sir fmadam know proposal might surprise e...,dear sir fmadam know proposal might surprise e...,dear sir fmadam know proposal might surprise e...
4,fyi,0,fyi,fyi,fyi,fyi,fyi


In [76]:
data = data.drop(["text", "email_cleaned", "email_cleaned_no_stopwords", "lemmas", "no_html"], axis=1)
data = data[data["preprocessed_text"] != ""]
data.head()

,label,preprocessed_text
0,1,dear sir strictly private business proposal mi...
2,0,noracheryl emailed dozen memo haiti weekend pl...
3,1,dear sir fmadam know proposal might surprise e...
4,0,fyi
5,0,sure bottom line need special security code ge...


In [77]:
data_train, data_val = train_test_split(data, train_size=0.8)

## Extra features

In [78]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,label,preprocessed_text,money_mark,suspicious_words,text_len
607,0,fyiforwarded message,0,0,20
523,1,auditorhead departmentbank scotlandunited king...,1,1,2171
277,0,abedin huma wednesday july pmsullivan jacob hfw,0,0,47
157,0,service post doc,0,0,16
189,0,interest hague latest briefing hostile tony to...,1,1,2115


## How would work the Bag of Words with Count Vectorizer concept?

In [79]:
X_train = data_train.drop(["label"], axis=1)
y_train = data_train["label"]

X_test = data_val.drop(["label"], axis=1)
y_test = data_val["label"]

In [80]:
# Count vectorizer builds a sparse matrix 
vectorizer.fit(X_train["preprocessed_text"])

X_train_bow = vectorizer.transform(X_train["preprocessed_text"])
X_test_bow = vectorizer.transform(X_test["preprocessed_text"])

X_train_bow.shape

(786, 23524)

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [81]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=24154) # Same number of features as in the bow

X_train_tfidf = tfidf.fit_transform(X_train["preprocessed_text"])
X_test_tfidf = tfidf.transform(X_test["preprocessed_text"])

top_idf_idx = tfidf.idf_.argsort()[-10:][::-1]
print([tfidf.get_feature_names_out()[id] for id in top_idf_idx])

['â½ã', 'hundredmillion', 'humastatementthis', 'humiliated', 'humiliatedbarack', 'humility', 'humilityi', 'hunderd', 'hundre', 'hundrend']


## And the Train a Classifier?

In [82]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr_bow = LogisticRegression(random_state=42)
lr_bow.fit(X_train_bow, y_train)
acc_bow = accuracy_score(y_test, lr_bow.predict(X_test_bow))
print(acc_bow)

0.9441624365482234


In [83]:
lr_tfidf = LogisticRegression(random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)
acc_tfidf = accuracy_score(y_test, lr_tfidf.predict(X_test_tfidf))
print(acc_tfidf)

0.934010152284264


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [84]:
from scipy.sparse import csr_matrix, hstack
from sklearn.naive_bayes import MultinomialNB

mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train)
acc_bow_mnb = accuracy_score(y_test, mnb_bow.predict(X_test_bow))
print(acc_bow_mnb)

mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)
acc_tfidf_mnb = accuracy_score(y_test, mnb_tfidf.predict(X_test_tfidf))
print(acc_tfidf_mnb)



extra_train = csr_matrix(X_train[["money_mark", "suspicious_words", "text_len"]].values)
X_train_bow_extra = hstack([X_train_bow, extra_train])
X_train_tfidf_extra = hstack([X_train_tfidf, extra_train])

extra_test = csr_matrix(X_test[["money_mark", "suspicious_words", "text_len"]].values)
X_test_bow_extra = hstack([X_test_bow, extra_test])
X_test_tfidf_extra = hstack([X_test_tfidf, extra_test])

mnb_bow_extra = MultinomialNB()
mnb_bow_extra.fit(X_train_bow_extra, y_train)
acc_bow_extra = accuracy_score(y_test, mnb_bow_extra.predict(X_test_bow_extra))
print(acc_bow_extra)

mnb_tfidf_extra = MultinomialNB()
mnb_tfidf_extra.fit(X_train_tfidf_extra, y_train)
acc_tfidf_extra = accuracy_score(y_test, mnb_tfidf_extra.predict(X_test_tfidf_extra))
print(acc_tfidf_extra)

0.9543147208121827
0.934010152284264
0.8730964467005076
0.6548223350253807


In [85]:
print("Bag of words + Logistic Regression")
print(acc_bow)
print("Bag of words + MultinomialNB")
print(acc_bow_mnb)
print("Bag of words + Extra features + MultinomialNB")
print(acc_bow_extra)
print("TF-IDF + Logistic Regression")
print(acc_tfidf)
print("TF-IDF + MultinomialNB")
print(acc_tfidf_mnb)
print("TF-IDF + Extra features + MultinomialNB")
print(acc_tfidf_extra)

Bag of words + Logistic Regression
0.9441624365482234
Bag of words + MultinomialNB
0.9543147208121827
Bag of words + Extra features + MultinomialNB
0.8730964467005076
TF-IDF + Logistic Regression
0.934010152284264
TF-IDF + MultinomialNB
0.934010152284264
TF-IDF + Extra features + MultinomialNB
0.6548223350253807


Extra Features make result worse

The best result -- bag of words + MultinomialNB